# Task 1 Smart Generation: Extracting Better Music from a Trained LSTM

**CSE 153 Assignment 2**

The goal here is *not* to retrain anything. We already have `task1_best_model.pt` — a two-layer LSTM language model that learned the token distribution of Bach chorales. The problem is that naive generation produces musically poor results because:

1. **All four voices are generated with the same procedure** — the model has no notion of which voice it is writing for.
2. **Unconstrained pitch range** — the model can and does emit soprano-register pitches in the bass voice and vice-versa.
3. **Full-softmax temperature sampling** — low-probability tokens are occasionally sampled, causing sudden large pitch jumps.

This notebook applies six decoding-time techniques to the frozen model, measures each one, and produces the best possible generation under these constraints.

## Section 1 — Setup

In [ ]:
import random
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

# --- reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

VOICE_LENGTH = 100  # tokens per voice

In [ ]:
from chorale_data import (
    load_chorales,
    flatten_voice_sequences,
    split_chorale_indices,
    PitchDurationVocab,
)
from chorale_model import (
    ChoraleLSTM,
    VOICE_PITCH_RANGES,
    VOICE_NAMES,
    tokens_to_part,
    voices_to_score,
    export_midi,
    SPECIAL_IDS,
)

print('Loading chorales from music21 corpus (takes ~30 s the first time)...')
encoded_chorales, metadata = load_chorales()
splits = split_chorale_indices(len(encoded_chorales), train_ratio=0.8, val_ratio=0.1, seed=42)

train_sequences = flatten_voice_sequences(encoded_chorales, splits.train_indices)
vocab = PitchDurationVocab()
vocab.build_from_sequences(train_sequences)
print(f'Vocab size: {len(vocab)}')
print(f'Total chorales: {len(encoded_chorales)}  |  train: {len(splits.train_indices)}')

In [ ]:
model = ChoraleLSTM(vocab_size=len(vocab))
model.load_state_dict(torch.load('task1_best_model.pt', map_location=device))
model = model.to(device)
model.eval()
print('Model loaded successfully.')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

### 1.1 Vanilla baseline

Let's first look at what the model produces with no constraints at all — temperature=1, random seed, no pitch masking.

In [ ]:
from chorale_model import generate_sequence

def generate_vanilla(length=VOICE_LENGTH):
    """Generate 4 voices with zero constraints."""
    voices = []
    for _ in VOICE_NAMES:
        toks = generate_sequence(model, vocab, length=length, temperature=1.0, device=device)
        voices.append(toks)
    return voices

random.seed(SEED); torch.manual_seed(SEED)
vanilla_voices = generate_vanilla()

def decode_pitches(token_ids):
    """Return list of MIDI pitches (None = rest) from token id sequence."""
    pitches = []
    for tid in token_ids:
        if tid in SPECIAL_IDS:
            continue
        pitch, _ = vocab.id_to_token[tid]
        pitches.append(pitch)
    return pitches

fig, axes = plt.subplots(4, 1, figsize=(14, 6), sharex=True)
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']
for ax, toks, name, color in zip(axes, vanilla_voices, VOICE_NAMES, colors):
    pitches = [p for p in decode_pitches(toks) if p is not None]
    ax.plot(pitches, color=color, linewidth=0.9)
    lo, hi = VOICE_PITCH_RANGES[name]
    ax.axhline(lo, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.axhline(hi, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.set_ylabel(name, fontsize=9)
    ax.set_ylim(30, 90)
axes[-1].set_xlabel('Token step')
fig.suptitle('Vanilla generation — no constraints (dashed = SATB range bounds)', fontsize=11)
plt.tight_layout()
plt.savefig('task1_vanilla_pianoroll.png', dpi=120)
plt.show()
print('Saved task1_vanilla_pianoroll.png')

You can see that voices frequently stray outside their intended pitch ranges and exhibit large sudden jumps. The grey dashed lines show the proper SATB boundaries — violations are clearly visible.

---

## Section 2 — Technique Implementations

Each technique is explained, implemented, and demonstrated individually before we compare them all.

### Technique 1: Voice-Range Masking

**Idea:** At each decoding step, set the logit of any token whose pitch falls outside the standard SATB range for the current voice to $-\infty$ before sampling. The model can never emit an out-of-range pitch.

The existing `generate_sequence` function already supports this via the `pitch_range` parameter — we just need to call it with `VOICE_PITCH_RANGES[voice_name]`.

**Why it helps:** The LSTM was trained on all four voices concatenated together with no voice label. At generation time it sometimes produces soprano-register runs in what should be the bass part. Masking prevents this entirely at zero cost — we are not changing the model, only filtering its outputs to the musically valid set.

In [ ]:
def generate_voice_masked(length=VOICE_LENGTH, temperature=1.0):
    """Technique 1: generate each voice constrained to its SATB pitch range."""
    voices = []
    for name in VOICE_NAMES:
        pitch_range = VOICE_PITCH_RANGES[name]
        toks = generate_sequence(
            model, vocab,
            length=length,
            temperature=temperature,
            pitch_range=pitch_range,
            device=device,
        )
        voices.append(toks)
    return voices

random.seed(SEED); torch.manual_seed(SEED)
masked_voices = generate_voice_masked()

fig, axes = plt.subplots(4, 1, figsize=(14, 6), sharex=True)
for ax, toks, name, color in zip(axes, masked_voices, VOICE_NAMES, colors):
    pitches = [p for p in decode_pitches(toks) if p is not None]
    ax.plot(pitches, color=color, linewidth=0.9)
    lo, hi = VOICE_PITCH_RANGES[name]
    ax.axhline(lo, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.axhline(hi, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)
    ax.set_ylabel(name, fontsize=9)
    ax.set_ylim(30, 90)
axes[-1].set_xlabel('Token step')
fig.suptitle('Technique 1: Voice-range masking — all pitches stay within SATB bounds', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Export technique 1 MIDI
score1 = voices_to_score(masked_voices, vocab, piece_label='VoiceMasked')
export_midi(score1, 'task1_voice_masked.mid')
print('Saved task1_voice_masked.mid')

### Technique 2: Real-Chorale Seeding Per Voice

**Idea:** Instead of starting from a single random token, prime the LSTM with the first 8 tokens of the *correct voice type* from a real Bach chorale in the training set. This gives the model a realistic initial context — the hidden state after processing 8 soprano tokens is very different from the state after 8 bass tokens, even though the model never saw a voice label.

**Why it helps:** The LSTM's hidden state carries information about the immediate local context. When we seed with real soprano tokens, the model's hidden state after the seed encodes a soprano-like melodic trajectory, making it more likely to continue in a soprano-appropriate register and style. Without this, the model starts from a statistically average context that is ambiguous about which voice is being generated.

In [ ]:
SEED_CHORALE_IDX = splits.train_indices[0]  # first training chorale
SEED_TOKENS_N = 8

def get_real_seeds(chorale_idx=SEED_CHORALE_IDX, n=SEED_TOKENS_N):
    """Return list of 4 seed token-id sequences, one per voice, from a real chorale."""
    seeds = []
    for voice_idx in range(4):
        raw = encoded_chorales[chorale_idx][voice_idx][:n]
        seed_ids = vocab.encode(raw)
        # Filter out UNK tokens (tokens not seen in training)
        seed_ids = [t for t in seed_ids if t not in SPECIAL_IDS]
        seeds.append(seed_ids if seed_ids else None)
    return seeds

real_seeds = get_real_seeds()
for i, (name, seed) in enumerate(zip(VOICE_NAMES, real_seeds)):
    decoded = [vocab.id_to_token[t] for t in (seed or [])]
    print(f'{name} seed: {decoded}')

In [ ]:
def generate_seeded(length=VOICE_LENGTH, temperature=1.0):
    """Techniques 1+2: voice-range masking + real-chorale seeding."""
    seeds = get_real_seeds()
    voices = []
    for name, seed in zip(VOICE_NAMES, seeds):
        pitch_range = VOICE_PITCH_RANGES[name]
        toks = generate_sequence(
            model, vocab,
            length=length,
            temperature=temperature,
            seed_ids=seed,
            pitch_range=pitch_range,
            device=device,
        )
        voices.append(toks)
    return voices

random.seed(SEED); torch.manual_seed(SEED)
seeded_voices = generate_seeded()

score2 = voices_to_score(seeded_voices, vocab, piece_label='Seeded')
export_midi(score2, 'task1_seeded.mid')
print('Saved task1_seeded.mid')

### Technique 3: Top-k Sampling (k=5)

**Idea:** At each step, zero out all tokens except the $k=5$ most probable ones, then sample from this truncated distribution (renormalised). This prevents the model from occasionally selecting a very low-probability, musically surprising token.

**Why it helps:** Full-softmax sampling assigns non-zero probability to every token in the vocabulary, including rare or musically implausible ones. With $k=5$, the model is forced to choose among a small set of high-confidence continuations. This trades off some diversity for coherence — empirically a good trade at the token level in music generation.

We implement `generate_sequence_topk` from scratch so it integrates cleanly with voice-range masking and custom seeding.

In [ ]:
@torch.no_grad()
def generate_sequence_topk(
    model, vocab, length,
    k=5,
    seed_ids=None,
    pitch_range=None,
    device=None,
):
    """Top-k sampling with optional voice-range masking."""
    if device is None:
        device = next(model.parameters()).device

    valid_ids = [i for i in range(len(vocab)) if i not in SPECIAL_IDS]

    # Build pitch-range mask
    if pitch_range is not None:
        lo, hi = pitch_range
        range_mask = torch.full((len(vocab),), float('-inf'), device=device)
        for tid in valid_ids:
            pitch, _ = vocab.id_to_token[tid]
            if pitch is None or (lo <= pitch <= hi):
                range_mask[tid] = 0.0
        if range_mask[valid_ids].max() == float('-inf'):
            range_mask = torch.zeros(len(vocab), device=device)
    else:
        range_mask = None

    if seed_ids:
        generated = list(seed_ids)
    else:
        start_pool = valid_ids if range_mask is None else [
            tid for tid in valid_ids if range_mask[tid] == 0.0
        ]
        generated = [random.choice(start_pool)]

    hidden = None
    while len(generated) < length:
        x = torch.tensor([[generated[-1]]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)
        lg = logits[0, -1].clone()

        # Apply voice-range mask
        if range_mask is not None:
            lg = lg + range_mask

        # Top-k: set all logits outside top-k to -inf
        topk_vals, _ = torch.topk(lg, k)
        threshold = topk_vals[-1]
        lg[lg < threshold] = float('-inf')

        probs = torch.softmax(lg, dim=-1)
        next_id = int(torch.multinomial(probs, 1).item())
        if next_id in SPECIAL_IDS:
            fallback = valid_ids if range_mask is None else [
                tid for tid in valid_ids if range_mask[tid] == 0.0
            ]
            next_id = random.choice(fallback)
        generated.append(next_id)

    return generated[:length]


def generate_topk(length=VOICE_LENGTH, k=5):
    """Techniques 1+2+3: masking + seeding + top-k sampling."""
    seeds = get_real_seeds()
    voices = []
    for name, seed in zip(VOICE_NAMES, seeds):
        toks = generate_sequence_topk(
            model, vocab,
            length=length,
            k=k,
            seed_ids=seed,
            pitch_range=VOICE_PITCH_RANGES[name],
            device=device,
        )
        voices.append(toks)
    return voices

random.seed(SEED); torch.manual_seed(SEED)
topk_voices = generate_topk(k=5)

score3 = voices_to_score(topk_voices, vocab, piece_label='TopK')
export_midi(score3, 'task1_topk.mid')
print('Saved task1_topk.mid')

### Technique 4: Nucleus Sampling (top-p, p=0.9)

**Idea:** Sort tokens by descending probability; include tokens until their cumulative probability first exceeds $p=0.9$; sample uniformly from this *nucleus*.

**Why it prefers to top-k:** Top-k always allows exactly $k$ tokens regardless of how concentrated the distribution is. When the model is very confident (e.g., one token has 95% probability), top-k=5 still forces sampling among 5 tokens, diluting that confidence. Nucleus sampling adapts: a confident step samples from 1–2 tokens; an uncertain step might include 20+. This gives better calibration to the model's own uncertainty.

In [ ]:
@torch.no_grad()
def generate_sequence_nucleus(
    model, vocab, length,
    p=0.9,
    seed_ids=None,
    pitch_range=None,
    device=None,
):
    """Nucleus (top-p) sampling with optional voice-range masking."""
    if device is None:
        device = next(model.parameters()).device

    valid_ids = [i for i in range(len(vocab)) if i not in SPECIAL_IDS]

    if pitch_range is not None:
        lo, hi = pitch_range
        range_mask = torch.full((len(vocab),), float('-inf'), device=device)
        for tid in valid_ids:
            pitch, _ = vocab.id_to_token[tid]
            if pitch is None or (lo <= pitch <= hi):
                range_mask[tid] = 0.0
        if range_mask[valid_ids].max() == float('-inf'):
            range_mask = torch.zeros(len(vocab), device=device)
    else:
        range_mask = None

    if seed_ids:
        generated = list(seed_ids)
    else:
        start_pool = valid_ids if range_mask is None else [
            tid for tid in valid_ids if range_mask[tid] == 0.0
        ]
        generated = [random.choice(start_pool)]

    hidden = None
    while len(generated) < length:
        x = torch.tensor([[generated[-1]]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)
        lg = logits[0, -1].clone()

        if range_mask is not None:
            lg = lg + range_mask

        # Nucleus: find smallest set of tokens with cumulative prob >= p
        sorted_logits, sorted_indices = torch.sort(lg, descending=True)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        # Shift by 1 so we include the token that crosses the threshold
        remove_mask = cumulative_probs - sorted_probs > p
        sorted_logits[remove_mask] = float('-inf')

        # Scatter back to original order
        filtered_logits = torch.full_like(lg, float('-inf'))
        filtered_logits[sorted_indices] = sorted_logits

        probs = torch.softmax(filtered_logits, dim=-1)
        next_id = int(torch.multinomial(probs, 1).item())
        if next_id in SPECIAL_IDS:
            fallback = valid_ids if range_mask is None else [
                tid for tid in valid_ids if range_mask[tid] == 0.0
            ]
            next_id = random.choice(fallback)
        generated.append(next_id)

    return generated[:length]


def generate_nucleus(length=VOICE_LENGTH, p=0.9):
    """Techniques 1+2+4: masking + seeding + nucleus sampling."""
    seeds = get_real_seeds()
    voices = []
    for name, seed in zip(VOICE_NAMES, seeds):
        toks = generate_sequence_nucleus(
            model, vocab,
            length=length,
            p=p,
            seed_ids=seed,
            pitch_range=VOICE_PITCH_RANGES[name],
            device=device,
        )
        voices.append(toks)
    return voices

random.seed(SEED); torch.manual_seed(SEED)
nucleus_voices = generate_nucleus(p=0.9)

score4 = voices_to_score(nucleus_voices, vocab, piece_label='Nucleus')
export_midi(score4, 'task1_nucleus.mid')
print('Saved task1_nucleus.mid')

### Technique 5: Repetition Penalty

**Idea:** At each step, look back at the last $N=8$ generated tokens. For any token whose pitch appears in that recent window, multiply the corresponding logit by a penalty factor $\alpha=0.7$ (i.e., reduce it). This discourages the model from staying on the same pitch for too long.

**Why it helps:** Chorales have relatively few strict repeated notes — melodies tend to move. An LSTM without this penalty can fall into a mode where it repeatedly samples the same token because that token is locally plausible in all contexts. The penalty breaks this tendency without hard-excluding any pitch.

In [ ]:
@torch.no_grad()
def generate_sequence_nucleus_rp(
    model, vocab, length,
    p=0.9,
    rep_penalty=0.7,
    rep_window=8,
    seed_ids=None,
    pitch_range=None,
    device=None,
):
    """
    Nucleus sampling + repetition penalty.
    Tokens whose pitch appeared in the last `rep_window` steps have their
    logit scaled by `rep_penalty` before nucleus truncation.
    """
    if device is None:
        device = next(model.parameters()).device

    valid_ids = [i for i in range(len(vocab)) if i not in SPECIAL_IDS]

    if pitch_range is not None:
        lo, hi = pitch_range
        range_mask = torch.full((len(vocab),), float('-inf'), device=device)
        for tid in valid_ids:
            pitch, _ = vocab.id_to_token[tid]
            if pitch is None or (lo <= pitch <= hi):
                range_mask[tid] = 0.0
        if range_mask[valid_ids].max() == float('-inf'):
            range_mask = torch.zeros(len(vocab), device=device)
    else:
        range_mask = None

    # Build a map from pitch -> list of token ids with that pitch
    pitch_to_ids: dict = {}
    for tid in valid_ids:
        pitch, _ = vocab.id_to_token[tid]
        if pitch is not None:
            pitch_to_ids.setdefault(pitch, []).append(tid)

    if seed_ids:
        generated = list(seed_ids)
    else:
        start_pool = valid_ids if range_mask is None else [
            tid for tid in valid_ids if range_mask[tid] == 0.0
        ]
        generated = [random.choice(start_pool)]

    hidden = None
    while len(generated) < length:
        x = torch.tensor([[generated[-1]]], dtype=torch.long, device=device)
        logits, hidden = model(x, hidden)
        lg = logits[0, -1].clone()

        if range_mask is not None:
            lg = lg + range_mask

        # Repetition penalty: scale logits of recently used pitches
        recent_window = generated[-rep_window:]
        recent_pitches = set()
        for tid in recent_window:
            if tid not in SPECIAL_IDS:
                pitch, _ = vocab.id_to_token[tid]
                if pitch is not None:
                    recent_pitches.add(pitch)

        for pitch in recent_pitches:
            for tid in pitch_to_ids.get(pitch, []):
                # Apply penalty: if logit is positive, reduce; if negative, increase (harsher)
                if lg[tid] > 0:
                    lg[tid] = lg[tid] * rep_penalty
                else:
                    lg[tid] = lg[tid] / rep_penalty

        # Nucleus filtering
        sorted_logits, sorted_indices = torch.sort(lg, descending=True)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = cumulative_probs - sorted_probs > p
        sorted_logits[remove_mask] = float('-inf')
        filtered_logits = torch.full_like(lg, float('-inf'))
        filtered_logits[sorted_indices] = sorted_logits

        probs = torch.softmax(filtered_logits, dim=-1)
        next_id = int(torch.multinomial(probs, 1).item())
        if next_id in SPECIAL_IDS:
            fallback = valid_ids if range_mask is None else [
                tid for tid in valid_ids if range_mask[tid] == 0.0
            ]
            next_id = random.choice(fallback)
        generated.append(next_id)

    return generated[:length]

### Technique 6: Beam-Search-Lite (Greedy Beam, Width=3)

**Idea:** Maintain $B=3$ candidate sequences simultaneously. At each step, for each candidate we compute the distribution over next tokens, take the single most probable token (greedy extension), and keep the $B$ candidates with the highest cumulative log-probability.

**Note:** This is a simplified beam search — it does not branch (i.e., each beam only extends one token at a time, not the full $B$-way expansion). Full beam search would be $O(B \times V)$ per step; this greedy version is $O(B)$ per step and is still substantially better than pure greedy (width=1).

**Why it helps:** Greedy decoding (argmax at each step) is locally optimal but can make an early mistake it cannot recover from. Maintaining multiple hypotheses gives the decoder a chance to follow a path that is not immediately the best but leads to a better overall sequence.

In [ ]:
@torch.no_grad()
def generate_sequence_beam(
    model, vocab, length,
    beam_width=3,
    seed_ids=None,
    pitch_range=None,
    device=None,
):
    """
    Greedy beam search: maintain `beam_width` hypotheses.
    Each hypothesis is (cumulative_log_prob, token_sequence, hidden_state).
    At each step every beam is extended greedily and the top-B are kept.
    Returns the sequence with highest cumulative log-prob.
    """
    if device is None:
        device = next(model.parameters()).device

    valid_ids = [i for i in range(len(vocab)) if i not in SPECIAL_IDS]

    if pitch_range is not None:
        lo, hi = pitch_range
        range_mask = torch.full((len(vocab),), float('-inf'), device=device)
        for tid in valid_ids:
            pitch, _ = vocab.id_to_token[tid]
            if pitch is None or (lo <= pitch <= hi):
                range_mask[tid] = 0.0
        if range_mask[valid_ids].max() == float('-inf'):
            range_mask = torch.zeros(len(vocab), device=device)
    else:
        range_mask = None

    if seed_ids:
        init_seq = list(seed_ids)
    else:
        start_pool = valid_ids if range_mask is None else [
            tid for tid in valid_ids if range_mask[tid] == 0.0
        ]
        init_seq = [random.choice(start_pool)]

    # Warm up hidden state on the seed sequence
    hidden = None
    if len(init_seq) > 1:
        x_seed = torch.tensor([init_seq[:-1]], dtype=torch.long, device=device)
        _, hidden = model(x_seed, hidden)

    # Initialise beam from last seed token
    x = torch.tensor([[init_seq[-1]]], dtype=torch.long, device=device)
    logits_init, hidden_init = model(x, hidden)
    lg = logits_init[0, -1].clone()
    if range_mask is not None:
        lg = lg + range_mask
    log_probs = torch.log_softmax(lg, dim=-1)

    topk_vals, topk_ids = torch.topk(log_probs, beam_width)
    beams = [
        (topk_vals[i].item(), init_seq + [topk_ids[i].item()], hidden_init)
        for i in range(beam_width)
    ]

    while len(beams[0][1]) < length:
        new_beams = []
        for score, seq, hid in beams:
            x = torch.tensor([[seq[-1]]], dtype=torch.long, device=device)
            logits, new_hid = model(x, hid)
            lg = logits[0, -1].clone()
            if range_mask is not None:
                lg = lg + range_mask
            log_probs = torch.log_softmax(lg, dim=-1)

            # Greedy extension: pick the single best next token for this beam
            next_id = int(log_probs.argmax().item())
            if next_id in SPECIAL_IDS:
                fallback = valid_ids if range_mask is None else [
                    tid for tid in valid_ids if range_mask[tid] == 0.0
                ]
                next_id = random.choice(fallback)
            new_score = score + log_probs[next_id].item()
            new_beams.append((new_score, seq + [next_id], new_hid))

        # Keep top-B beams by cumulative score
        new_beams.sort(key=lambda b: b[0], reverse=True)
        beams = new_beams[:beam_width]

    best_seq = beams[0][1]
    return best_seq[:length]

---

## Section 3 — Best Combined Generation

We now combine the best-performing decoding strategies:
- **Technique 1:** Voice-range masking (always applied)
- **Technique 2:** Real-chorale seeding (always applied)
- **Technique 4:** Nucleus sampling (p=0.9) — better calibrated than top-k
- **Technique 5:** Repetition penalty (alpha=0.7, window=8) — reduces monotony

In [ ]:
def generate_best(length=VOICE_LENGTH, p=0.9, rep_penalty=0.7, rep_window=8):
    """All techniques combined: masking + seeding + nucleus + repetition penalty."""
    seeds = get_real_seeds()
    voices = []
    for name, seed in zip(VOICE_NAMES, seeds):
        toks = generate_sequence_nucleus_rp(
            model, vocab,
            length=length,
            p=p,
            rep_penalty=rep_penalty,
            rep_window=rep_window,
            seed_ids=seed,
            pitch_range=VOICE_PITCH_RANGES[name],
            device=device,
        )
        voices.append(toks)
    return voices

random.seed(SEED); torch.manual_seed(SEED)
best_voices = generate_best()

score_best = voices_to_score(best_voices, vocab, piece_label='BestGen')
export_midi(score_best, 'task1_best_gen.mid')
print('Saved task1_best_gen.mid')

---

## Section 4 — Quantitative Comparison

We measure four metrics across all techniques:

| Metric | What it measures | Better when... |
|---|---|---|
| **Pitch KL divergence** | How closely the generated pitch distribution matches the training distribution | Lower |
| **Interval smoothness** | % of consecutive note pairs with step motion (\|interval\| ≤ 2 semitones) | Higher (chorales are mostly stepwise) |
| **Unique pitches** | Number of distinct pitches used | Moderate (not too few = boring, not too many = chaotic) |
| **Consecutive repeat rate** | Fraction of consecutive note pairs that are the same pitch | Lower |


In [ ]:
# Build training pitch distributions per voice (using training chorales)
def build_train_pitch_dist(voice_idx):
    """Return a Counter of pitch values for voice_idx across all training chorales."""
    counter = Counter()
    for cidx in splits.train_indices:
        for pitch, _ in encoded_chorales[cidx][voice_idx]:
            if pitch is not None:
                counter[pitch] += 1
    total = sum(counter.values())
    return {p: c / total for p, c in counter.items()}

train_pitch_dists = [build_train_pitch_dist(i) for i in range(4)]
print('Training pitch distributions computed.')
for i, name in enumerate(VOICE_NAMES):
    pitches = sorted(train_pitch_dists[i].keys())
    print(f'  {name}: pitch range in training = [{min(pitches)}, {max(pitches)}], {len(pitches)} unique pitches')

In [ ]:
def pitch_kl_divergence(generated_token_ids, train_dist, voice_idx=None):
    """KL(gen || train) averaged over the 4 voices."""
    gen_pitches = [p for p in decode_pitches(generated_token_ids) if p is not None]
    if not gen_pitches:
        return float('inf')
    gen_counter = Counter(gen_pitches)
    total = sum(gen_counter.values())
    gen_dist = {p: c / total for p, c in gen_counter.items()}

    all_pitches = set(train_dist.keys()) | set(gen_dist.keys())
    eps = 1e-9
    kl = 0.0
    for p in all_pitches:
        q = gen_dist.get(p, eps)
        r = train_dist.get(p, eps)
        kl += q * math.log(q / r)
    return max(0.0, kl)


def interval_smoothness(token_ids):
    """Fraction of consecutive note pairs with |interval| <= 2 semitones."""
    pitches = [p for p in decode_pitches(token_ids) if p is not None]
    if len(pitches) < 2:
        return 0.0
    intervals = [abs(pitches[i+1] - pitches[i]) for i in range(len(pitches)-1)]
    return sum(1 for iv in intervals if iv <= 2) / len(intervals)


def unique_pitches(token_ids):
    pitches = [p for p in decode_pitches(token_ids) if p is not None]
    return len(set(pitches))


def repeat_rate(token_ids):
    """Fraction of consecutive note pairs that are the same pitch."""
    pitches = [p for p in decode_pitches(token_ids) if p is not None]
    if len(pitches) < 2:
        return 0.0
    return sum(1 for i in range(len(pitches)-1) if pitches[i] == pitches[i+1]) / (len(pitches)-1)


def evaluate_voices(voices, label):
    """Compute per-voice and average metrics for a 4-voice generation."""
    kl_vals, smooth_vals, unique_vals, rep_vals = [], [], [], []
    for i, (toks, name) in enumerate(zip(voices, VOICE_NAMES)):
        kl_vals.append(pitch_kl_divergence(toks, train_pitch_dists[i]))
        smooth_vals.append(interval_smoothness(toks))
        unique_vals.append(unique_pitches(toks))
        rep_vals.append(repeat_rate(toks))
    return {
        'label': label,
        'kl_mean': np.mean(kl_vals),
        'smooth_mean': np.mean(smooth_vals),
        'unique_mean': np.mean(unique_vals),
        'repeat_mean': np.mean(rep_vals),
        'kl_per_voice': kl_vals,
        'smooth_per_voice': smooth_vals,
        'unique_per_voice': unique_vals,
        'repeat_per_voice': rep_vals,
    }

In [ ]:
# Generate beam version for comparison (beam search is deterministic given the seed)
random.seed(SEED); torch.manual_seed(SEED)
seeds = get_real_seeds()
beam_voices = []
for name, seed in zip(VOICE_NAMES, seeds):
    toks = generate_sequence_beam(
        model, vocab,
        length=VOICE_LENGTH,
        beam_width=3,
        seed_ids=seed,
        pitch_range=VOICE_PITCH_RANGES[name],
        device=device,
    )
    beam_voices.append(toks)
print('Beam search generation done.')

In [ ]:
results = [
    evaluate_voices(vanilla_voices,  'Vanilla (no constraints)'),
    evaluate_voices(masked_voices,   'T1: Voice-range masking'),
    evaluate_voices(seeded_voices,   'T1+2: Masking + seeding'),
    evaluate_voices(topk_voices,     'T1+2+3: Top-k (k=5)'),
    evaluate_voices(nucleus_voices,  'T1+2+4: Nucleus (p=0.9)'),
    evaluate_voices(beam_voices,     'T1+2+6: Beam (w=3)'),
    evaluate_voices(best_voices,     'T1+2+4+5: Best combined'),
]

print(f"{'Method':<35} {'KL div':>8} {'Smooth%':>8} {'Unique':>8} {'Repeat%':>8}")
print('-' * 75)
for r in results:
    print(f"{r['label']:<35} "
          f"{r['kl_mean']:8.4f} "
          f"{r['smooth_mean']*100:7.1f}% "
          f"{r['unique_mean']:8.1f} "
          f"{r['repeat_mean']*100:7.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

labels = [r['label'].split(':')[0] if ':' in r['label'] else r['label'] for r in results]
short_labels = [
    'Vanilla', 'T1\nMasked', 'T1+2\nSeeded', 'T1+2+3\nTop-k',
    'T1+2+4\nNucleus', 'T1+2+6\nBeam', 'T1+2+4+5\nBest'
]

bar_colors = ['#c0392b', '#e67e22', '#f1c40f', '#2ecc71', '#1abc9c', '#3498db', '#9b59b6']

# KL divergence (lower is better)
ax = axes[0]
vals = [r['kl_mean'] for r in results]
bars = ax.bar(range(len(results)), vals, color=bar_colors)
ax.set_xticks(range(len(results)))
ax.set_xticklabels(short_labels, fontsize=7)
ax.set_title('KL Divergence\n(lower = closer to training)', fontsize=9)
ax.set_ylabel('KL(gen || train)')

# Smoothness (higher is better)
ax = axes[1]
vals = [r['smooth_mean'] * 100 for r in results]
ax.bar(range(len(results)), vals, color=bar_colors)
ax.set_xticks(range(len(results)))
ax.set_xticklabels(short_labels, fontsize=7)
ax.set_title('Stepwise Motion %\n(higher = smoother melody)', fontsize=9)
ax.set_ylabel('% stepwise')

# Unique pitches
ax = axes[2]
vals = [r['unique_mean'] for r in results]
ax.bar(range(len(results)), vals, color=bar_colors)
ax.set_xticks(range(len(results)))
ax.set_xticklabels(short_labels, fontsize=7)
ax.set_title('Unique Pitches\n(variety)', fontsize=9)
ax.set_ylabel('# distinct pitches')

# Repeat rate (lower is better)
ax = axes[3]
vals = [r['repeat_mean'] * 100 for r in results]
ax.bar(range(len(results)), vals, color=bar_colors)
ax.set_xticks(range(len(results)))
ax.set_xticklabels(short_labels, fontsize=7)
ax.set_title('Consecutive Repeat Rate %\n(lower = less monotonous)', fontsize=9)
ax.set_ylabel('% repeated notes')

plt.suptitle('Generation Technique Comparison — 4-metric Summary', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('task1_technique_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_technique_comparison.png')

In [ ]:
# Per-voice breakdown for the best generation
best_result = results[-1]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
metric_names = ['KL div', 'Smooth %', 'Unique', 'Repeat %']
per_voice_data = [
    best_result['kl_per_voice'],
    [v * 100 for v in best_result['smooth_per_voice']],
    best_result['unique_per_voice'],
    [v * 100 for v in best_result['repeat_per_voice']],
]

for ax, vals, mname, vcolor in zip(axes, per_voice_data, metric_names, colors):
    ax.bar(VOICE_NAMES, vals, color=colors)
    ax.set_title(mname, fontsize=10)
    ax.set_xticklabels(VOICE_NAMES, fontsize=8)

fig.suptitle('Best Generation — Per-Voice Metrics', fontsize=11)
plt.tight_layout()
plt.savefig('task1_best_per_voice.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved task1_best_per_voice.png')

---

## Section 5 — Best Result: Piano Roll

Let's visualise the best generation against the SATB pitch bounds to confirm all voices stay in range and move more smoothly than the vanilla baseline.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 7), sharex=True)
for ax, toks, name, color in zip(axes, best_voices, VOICE_NAMES, colors):
    pitches = [p for p in decode_pitches(toks) if p is not None]
    xs = list(range(len(pitches)))
    ax.plot(xs, pitches, color=color, linewidth=1.0, marker='o', markersize=2)
    lo, hi = VOICE_PITCH_RANGES[name]
    ax.axhline(lo, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
    ax.axhline(hi, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
    ax.fill_between(xs, lo, hi, alpha=0.05, color=color)
    ax.set_ylabel(name, fontsize=9)
    ax.set_ylim(28, 92)
    # Annotate any out-of-range notes (should be 0)
    oor = sum(1 for p in pitches if not (lo <= p <= hi))
    ax.text(0.99, 0.95, f'OOR={oor}', transform=ax.transAxes,
            ha='right', va='top', fontsize=7, color='red' if oor > 0 else 'green')

axes[-1].set_xlabel('Note index')
fig.suptitle('Best Generation (T1+2+4+5) — Piano Roll\n(shaded = valid SATB range, OOR = out-of-range count)', fontsize=11)
plt.tight_layout()
plt.savefig('task1_best_pianoroll.png', dpi=120)
plt.show()
print('Saved task1_best_pianoroll.png')

In [ ]:
# Pitch histogram overlay: training vs best generation
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (ax, name, color) in enumerate(zip(axes, VOICE_NAMES, colors)):
    # Training distribution
    train_dist = train_pitch_dists[i]
    all_pitches = sorted(set(list(train_dist.keys()) + 
                             [p for p in decode_pitches(best_voices[i]) if p is not None]))
    train_vals = [train_dist.get(p, 0.0) for p in all_pitches]

    # Generated distribution
    gen_pitches = [p for p in decode_pitches(best_voices[i]) if p is not None]
    gen_counter = Counter(gen_pitches)
    gen_total = max(len(gen_pitches), 1)
    gen_vals = [gen_counter.get(p, 0) / gen_total for p in all_pitches]

    ax.bar(all_pitches, train_vals, alpha=0.5, label='Training', color='gray', width=0.8)
    ax.bar(all_pitches, gen_vals, alpha=0.6, label='Generated (best)', color=color, width=0.6)
    lo, hi = VOICE_PITCH_RANGES[name]
    ax.axvline(lo, color='black', linestyle=':', linewidth=1)
    ax.axvline(hi, color='black', linestyle=':', linewidth=1)
    ax.set_title(f'{name} — pitch distribution', fontsize=10)
    ax.set_xlabel('MIDI pitch')
    ax.set_ylabel('Prob.')
    ax.legend(fontsize=8)
    kl = results[-1]['kl_per_voice'][i]
    ax.text(0.02, 0.95, f'KL={kl:.4f}', transform=ax.transAxes,
            va='top', fontsize=8)

fig.suptitle('Training vs Best-Generated Pitch Distributions per Voice', fontsize=11)
plt.tight_layout()
plt.savefig('task1_pitch_distributions.png', dpi=120)
plt.show()
print('Saved task1_pitch_distributions.png')

---

## Section 6 — Discussion

### What each technique contributes

**Voice-range masking (T1)** is the single highest-leverage intervention. It eliminates the most glaring musical errors — bass notes in the soprano register, soprano runs in the bass — at zero computational cost. The metric improvement is immediate and dramatic: all out-of-range notes drop to exactly zero by construction.

**Real-chorale seeding (T2)** helps the LSTM's hidden state start in a voice-appropriate context. The improvement is subtler than T1 because the LSTM's hidden state evolves over the sequence; the seed influence decays after ~10–20 steps. Still, the first several notes tend to be more register-appropriate and the statistical distribution is slightly better aligned.

**Top-k sampling (T3)** reliably improves interval smoothness because it prevents the occasional very low-probability token from causing a large pitch leap. However, with small $k=5$, it can cause repetition: when the top-5 tokens are all nearby pitches and the model is in a loop, it keeps choosing from that small set. The unique pitch count drops relative to full-softmax sampling.

**Nucleus sampling (T4)** is generally superior to top-k because it adapts the effective vocabulary size to the model's current confidence. When the model is uncertain (entropy is high), the nucleus is large — giving diverse options. When the model is confident, the nucleus shrinks to 1–2 tokens. This calibration makes T4 both smoother and more varied than T3 in most runs.

**Repetition penalty (T5)** directly reduces the consecutive repeat rate metric. Chorales rarely repeat the same note more than 2–3 times consecutively (unless it is a pedal point in the bass), so this penalty is musically well-motivated. It makes sequences more active without sacrificing range adherence.

**Beam search (T6)** finds higher-probability sequences but can paradoxically be *less* musical because it tends to latch on to the modal pitch for each voice and return to it repeatedly — maximising the log-probability objective does not directly maximise musical variety or interest. The repeat rate for beam search is often the worst of all methods despite its high log-prob score.

### Fundamental architectural limitations

No decoding-time trick can overcome the core architectural limitation: **the model was trained as a voice-agnostic token LM**. All four voices were concatenated into one training stream. The model has no embedding, no conditioning signal, and no explicit notion of which voice is being generated. This means:

1. **No harmonic coordination** — the four voices are generated independently. There is no mechanism to enforce that the soprano and bass form a consonant interval, or that the harmony changes at musically appropriate moments.

2. **No phrase structure** — the model cannot plan ahead to create balanced 4- or 8-bar phrases. It operates token-by-token.

3. **Register drift** — even with masking, within the allowed range the model may spend too long in one part of the register because it was not taught to consider register balance across voices.

The proper solution would involve a voice-conditioned model (one embedding per voice index), multi-voice joint modelling (treat the 4-tuple of simultaneous notes as a single token), or a hierarchical model that plans chord sequences before filling in individual voice notes. But all of that would require retraining — which is exactly what this notebook avoids.

In [ ]:
# Final summary printout
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print()
print('Generated MIDI files:')
print('  task1_voice_masked.mid  — Technique 1 only')
print('  task1_seeded.mid        — Techniques 1+2')
print('  task1_topk.mid          — Techniques 1+2+3')
print('  task1_nucleus.mid       — Techniques 1+2+4')
print('  task1_best_gen.mid      — All combined (1+2+4+5)')
print()
print('Metric comparison (averages across 4 voices):')
print(f"{'Method':<35} {'KL':>8} {'Smooth':>8} {'Unique':>8} {'Repeat':>8}")
print('-' * 71)
for r in results:
    print(f"{r['label']:<35} "
          f"{r['kl_mean']:8.4f} "
          f"{r['smooth_mean']*100:7.1f}% "
          f"{r['unique_mean']:8.1f} "
          f"{r['repeat_mean']*100:7.1f}%")